### Rasterio
rasterio is a python library used for working with geopatial raster datasets like opening and manipulating the .tif files.
Raster formats store pixel-based spatial data, while GeoJSON stores vector geometry.
They interact through operations like masking, rasterization, and vectorization. But for them to work together their CRS must be same.
Raster CRS == GeoJSON CRS

In [33]:
import rasterio
dataset = rasterio.open("/Users/mohammadbilal/Documents/Projects/Practice/python/Rasterio/example.tif")
print(dataset.name)
print(dataset.mode) # mode of the file 
print(dataset.crs) # coordinate reference system
print("Number of bands:", dataset.count) # number of bands in the raster file
w, h = dataset.width, dataset.height
print(f"width: {w}, height: {h}")

/Users/mohammadbilal/Documents/Projects/Practice/python/Rasterio/example.tif
r
EPSG:4326
Number of bands: 1
width: 547, height: 421


In [ ]:
# printing the dataype of each band
src = rasterio.open("/Users/mohammadbilal/Documents/Projects/Practice/python/Rasterio/example2.tif")
bands = zip(src.indexes, src.dtypes)
for band, dtype in bands:
    print(f"  Band {band}: {dtype}")
    
band1 = src.read(1) # read the first band # print the values of the first band


  Band 1: float32
[[102.5  102.43 102.35 ...  98.6  101.67 100.48]
 [102.56 102.56 102.52 ... 101.21 101.57 100.55]
 [102.65 102.58 102.46 ... 104.05 102.85 103.64]
 ...
 [109.97 109.95 109.99 ... 136.94 136.85 135.61]
 [109.98 109.95 110.01 ... 132.11 130.36 129.03]
 [109.98 109.97 110.   ... 127.52 127.52 127.5 ]]


In [32]:
# calculating the area covered by the raster file
print(src.bounds)
l, b, r, t = src.bounds
area = (r - l) * (t - b) # area = width * height
print(f"Area covered by the raster file: {area} square units")

# calculating the pixel size
w, h = src.width, src.height
print(f"width: {w}, height: {h}")
pixel_width = (r - l) / w
pixel_height = (t - b) / h
print(f"Pixel size: {pixel_width} x {pixel_height} units")


BoundingBox(left=356000.0, bottom=5699000.0, right=357000.0, top=5700000.0)
Area covered by the raster file: 1000000.0 square units
width: 1000, height: 1000
Pixel size: 1.0 x 1.0 units


In [ ]:
src.transform # affine transformation coefficients (a, b, c, d, e, f) where:
# a: pixel width (size of a pixel in the x-direction)
# b: row rotation (typically 0)
# c: x-coordinate of the upper-left corner of the upper-left pixel
# d: column rotation (typically 0)
# e: pixel height (size of a pixel in the y-direction, typically negative)
# f: y-coordinate of the upper-left corner of the upper-left pixel

Affine(1.0, 0.0, 356000.0,
       0.0, -1.0, 5700000.0)

In [ ]:
# turning pixel coordinates to real-world coordinates

a, b, c, d, e, f = tuple(src.transform)[:6] 
x = 100 
y = 200
real_x = a * x + b * y + c  # affine transformation formula: real_x = a * x + b * y + c
real_y = d * x + e * y + f  

real_x_center = real_x + pixel_width / 2
real_y_center = real_y - pixel_height / 2
print(f"Real-world coordinates of the pixel: ({real_x_center}, {real_y_center})")

Real-world coordinates of the pixel: (356100.5, 5699799.5)


In [43]:
from rasterio.transform import xy
real_x, real_y = xy(src.transform, y, x) # note the order of y and x
print(f"Real-world coordinates of the pixel: ({real_x}, {real_y})")

Real-world coordinates of the pixel: (356100.5, 5699799.5)


In [47]:
band1 = src.read(1)
print(band1[0])

[102.5  102.43 102.35 102.32 102.24 102.19 102.56 102.07 101.94 101.98
 101.8  101.76 101.74 101.68 101.66 101.5  101.45 101.42 102.24 101.21
 103.11 104.82 112.26 112.43 113.37 113.05 112.13 109.65 100.39 100.39
 112.98 113.67 113.52 111.55 113.17 113.55 113.07 111.76 111.27 110.2
 106.47 111.19 111.37 114.46 114.27 115.09 115.27 114.55 115.87 114.77
 114.12 114.7  113.54 108.79 107.68 102.87  99.15  99.13  99.06  99.03
  98.98  98.95 104.49 104.89 108.51 103.14  98.94  98.9   98.84  98.83
  98.78  98.73  98.71  98.68  98.65  98.66  98.83  98.82  98.84  98.93
  99.53 102.03 100.83 102.38 102.46 101.98 107.52 109.47 110.01 111.06
 110.89 110.95 111.98 112.45 112.64 113.36 111.29 110.47 101.59 100.16
 100.17 100.17 100.18 100.17 100.17 100.18 100.19 100.22 100.25 100.28
 100.31 100.37 100.43 100.45 100.46 100.49 100.48 100.5  100.54 100.57
 100.6  100.63 100.64 100.73 100.82 100.83 101.4  100.84 102.02 101.02
 101.01 101.   100.92 100.89 100.9  100.96 100.93 100.94 101.09 101.05
 101.17

In [ ]:
# raster dataset is divided into smaller blocks of data for efficient reading and processing
print(src.block_shapes) 

[(3, 547)]
